# Assess clinical answers

**Goal:** Inspect the blinded rubric and reconstruct agreement-only paired inclusion.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Read the independent-judge rubric

Gemini 3.8 Flash and GLM 5.3 Flash assessed each answer independently. Dataset review by a sleep expert is distinct from these model-answer labels. Historical prompts retain the wording used at the time of judging; they are not rewritten to imply retrospective expert validation.

In [ ]:
display(read_json(ROOT / "data/config/clinical_scoring_rubric.json"))
import inspect
from sleepinn_study.clinical_judge import make_request
from sleepinn_study.glm_judge import request_for
print(inspect.getsource(make_request))
print(inspect.getsource(request_for))

## 3. Reconstruct response-level agreement

Both judges must give the same valid label: correct, partial or incorrect. Matching ungradable labels do not qualify. Correct=1 and partial/incorrect=0 for the strict endpoint; partial credit is a separate sensitivity endpoint.

In [ ]:
from sleepinn_study.statistics import consensus_scores
comparison = pd.read_csv(ROOT / "results/analysis/clinical/judge_comparison_19_systems.csv")
scored = consensus_scores(comparison)
assert len(scored) == 4560 and scored.groupby(["model_id", "precision"]).ngroups == 19
display(pd.crosstab(scored.gemini_label, scored.glm_label))
print("Responses with valid label agreement:", scored.included_exact_consensus.sum())

## 4. Retain complete case pairs

For the RAG contrast, the no-RAG and with-RAG answers must each have agreement. This produces different retained case counts across models. It is conditional agreement analysis, not accuracy on all 120 cases.

In [ ]:
pairs = scored.pivot(index=["model_id", "precision", "item_id"], columns="mode", values="strict_score")
eligible = pairs[["no_rag", "with_rag"]].dropna()
assert len(eligible) == 1831
display(eligible.groupby(["model_id", "precision"]).size().rename("retained case pairs"))
display(pd.read_csv(ROOT / "results/analysis/clinical/clinical_all_gradable_sensitivity.csv"))